# 11 — Design the next dataset project safely

**Estimated time:** 35 minutes<br>
**Prerequisites:** 10 — Capstone model versus hybrid<br>
**Learner-produced evidence:** a dataset-review checklist and task-specific evaluation plan

## Learning objectives

- Treat optional datasets as a review queue, not pre-approved inputs.
- Narrow the task before adding multilingual, extraction, transformation, or code-generation complexity.
- Match evaluation to the behavior instead of reusing intent metrics blindly.

This notebook is a teaching interface over the reusable code in `src/`.
It uses only prepared local files. Run `make prepare-flight` before the trip;
no cell installs packages or downloads data.


## Why this matters

The workflow transfers to new domains, but the dataset contract and evaluator do not transfer blindly. A support-intent metric is wrong for image documents, multi-label findings, code generation, or answers with many valid phrasings. This final notebook teaches how to identify the task before selecting data, models, and scores—and when to stop because the required authority or modality is missing.

## Key terms in plain language

- **task formulation:** a precise statement of the input, output, constraints, user decision, and failure costs.
- **input/output contract:** the machine- and reviewer-checkable shape and meaning of accepted inputs and outputs.
- **modality:** the type of signal required, such as text, image, audio, tabular data, or combinations.
- **OCR:** optical character recognition: converting pixels that depict text into machine-readable text, with possible errors.
- **multi-label task:** a task where more than one label can be correct for the same example.
- **exact match:** a strict metric requiring predicted and expected representations to be identical after defined normalization.
- **field-level F1:** precision/recall-based scoring over predicted versus expected fields or items.
- **executable evaluation:** checking generated code or structured plans by parsing, compiling, running tests, or enforcing invariants.
- **human calibration:** measuring and reconciling reviewer agreement so subjective labels or judges have a known interpretation.


## Mental model — how to think about this

Choose the evaluator from the output contract and failure cost, not from the previous notebook. Work backward: what decision will consume the output, what makes that output correct or safe, who can know the truth, what modality carries the evidence, and which automatic plus human checks approximate those facts? Only then choose a dataset, baseline, prompt, or tuning method.

### Running example

An invoice project may contain image files, OCR text, token labels, boxes, or only some of them. A text-only model cannot see pixels. First verify the actual modality and ground truth; then choose field-level extraction metrics, normalization checks, and human review. Reusing intent-classification macro F1 would answer the wrong question.

### Questions to ask before continuing

- Does the proposed input actually contain the information and modality needed for the target answer?
- Can multiple outputs be correct, and if so what semantic or executable check replaces naive exact match?
- Who supplies ground truth, and how will disagreement or uncertainty be represented?
- Do license, privacy, safety, or missing-domain facts block the project before model selection?


## Current best practices

**Guidance reviewed:** 2026-08-01. These are reasons to inspect future tool changes, not a claim that practice stops evolving.

- **Verify a real sample and its documentation first.** Confirm fields, modalities, license, label authority, and known limitations before designing an adapter around a dataset name.
- **Start with one narrow task.** A small explicit contract makes leakage analysis, baselines, error taxonomies, and promotion gates possible.
- **Match metrics to output semantics.** Use per-field or set metrics for structured/multi-label output, executable checks for code, retrieval measures for search, and calibrated human review for subjective quality.
- **Respect modality boundaries.** A text-only model does not inspect pixels; OCR or a vision-capable component must be named, versioned, and evaluated as part of the system.
- **Block unsuitable projects explicitly.** `Not verified`, `requires external authority`, and `not enough evidence` are successful design outcomes when they prevent an invalid experiment.

## Common mistakes and why they fail

- **Assuming columns from a dataset title or screenshot.** Inspect the actual versioned bytes and dataset card.
- **Expecting a text model to read scanned images.** Extracted text is a separate fallible component and needs its own evaluation.
- **Using exact match where several answers are valid.** Formatting variation can be penalized while substantive errors pass unnoticed.
- **Using an LLM judge without calibration.** Judges can be biased, inconsistent, or correlated with the model under test; compare with human labels.
- **Combining many tasks in the first experiment.** Mixed targets obscure which capability, data, and evaluator caused success or failure.

### What kind of guidance is this?

A **specification** defines a technical contract; **tool guidance** describes current official library behavior; **risk guidance** is voluntary governance guidance; and a **course rule** is this project's deliberately conservative choice. Do not call all four a formal standard. The lesson is complete offline; these primary links are optional follow-up reading.

- **Tool guidance:** [MLflow evaluation datasets and systematic evaluation](https://mlflow.org/docs/latest/genai/datasets/)
- **Specification:** [JSON Schema Draft 2020-12](https://json-schema.org/draft/2020-12)
- **Risk guidance:** [NIST AI RMF Generative AI Profile](https://www.nist.gov/publications/artificial-intelligence-risk-management-framework-generative-artificial-intelligence)


## Setup — run, do not edit

Run the next cell once. It verifies the dedicated local Python kernel, finds
this sample project, and enables supported offline flags **before** model or
tracking libraries are imported. A successful cell ends with `setup: ready`.

This is one defense layer, not proof that every native library is physically
incapable of networking. The flight-preparation manifest, cached assets,
socket-denial checks, and a Wi-Fi-off rehearsal provide the other layers.


In [ ]:
import sys
from importlib import import_module
from pathlib import Path

current = Path.cwd().resolve()
project_root = None
for candidate in (current, *current.parents):
    direct = candidate
    nested = candidate / "examples" / "local-finetuning"
    if (direct / "src" / "aai_local_finetuning").is_dir():
        project_root = direct
        break
    if (nested / "src" / "aai_local_finetuning").is_dir():
        project_root = nested
        break
if project_root is None:
    raise RuntimeError(
        "Cannot locate examples/local-finetuning. Open this notebook from the "
        "repository, or run `make notebook` from the repository root."
    )

expected_python = (project_root / ".venv" / "bin" / "python").resolve()
active_python = Path(sys.executable).resolve()
if not expected_python.is_file() or active_python != expected_python:
    raise RuntimeError(
        "Wrong notebook kernel. Run `make notebook` from the repository root, "
        "then select 'AAI Local Fine-Tuning (offline)'. "
        f"Active Python: {active_python}; expected: {expected_python}"
    )

source_root = str(project_root / "src")
if source_root not in sys.path:
    sys.path.insert(0, source_root)

enable_offline_environment = import_module(
    "aai_local_finetuning.offline"
).enable_offline_environment
enable_offline_environment()

{
    "setup": "ready",
    "kernel": "AAI Local Fine-Tuning (offline)",
    "python": str(active_python),
    "network_library_flags": "enabled",
    "note": "Continue to the lesson; this cell is setup, not an exercise.",
}

## Why these are plans, not executable labs yet

The optional Kaggle datasets are not cached or verified in this project.
Their current schema, license, access, source composition, sensitive-data
risk, and redistribution terms must be reviewed before code or claims are
added. Offline study should never fill those gaps with assumptions.


In [ ]:
candidates = [
    {
        "project": "multilingual support tickets",
        "first_task": "English-only queue/priority/type/review routing",
        "evaluation": "per-field metrics, imbalance and queue/priority slices",
        "status": "unsuitable until current source review is complete",
    },
    {
        "project": "invoice extraction",
        "first_task": "text-only fields only if reliable OCR text exists",
        "evaluation": "field exactness/F1, normalization, hallucinated fields",
        "status": "unsuitable until modality, schema, and rights are verified",
    },
    {
        "project": "prompt transformation",
        "first_task": "required prompt components and structured output",
        "evaluation": "rubric components plus calibrated human review",
        "status": "optional; open-ended scoring is less deterministic",
    },
    {
        "project": "MiniZinc generation",
        "first_task": "small natural-language-to-program problems",
        "evaluation": "parse, compile, execute, constraints, objective, runtime",
        "status": "advanced; may exceed tiny-model capability",
    },
]
candidates

## Required source review

A current review must record title, owner, URL, license, permitted use,
redistribution, size, formats, confirmed columns, languages, labels,
missingness, duplicates, sensitive information, human/synthetic origin,
accessibility, and access date. An unclear license means unsuitable until
the learner verifies it directly.


In [ ]:
required_review_fields = (
    "title",
    "owner",
    "current_url",
    "license",
    "permitted_use",
    "redistribution",
    "size",
    "formats",
    "source_modalities",
    "model_input_modality",
    "columns",
    "languages",
    "label_quality",
    "missing_values",
    "duplicate_rate",
    "sensitive_information",
    "record_origin",
    "accessible",
    "accessed_on",
)
required_review_fields

## Exercise — draft, but do not invent, a review

Choose one project. Every field carries an explicit `verified`, `unknown`,
or `blocked` state. A plausible placeholder is not verified evidence. The
gate stays false whenever rights, permitted use, redistribution, access,
schema, or modality is unknown or blocked.


In [ ]:
selected_project = "invoice extraction"
dataset_review = {
    field: {
        "state": "unknown",
        "value": None,
        "evidence": None,
    }
    for field in required_review_fields
}
dataset_review["title"]["value"] = "verify current title online"
blocking_fields = (
    "license",
    "permitted_use",
    "redistribution",
    "formats",
    "source_modalities",
    "model_input_modality",
    "columns",
    "accessible",
)
suitable_for_lab = all(
    dataset_review[field]["state"] == "verified"
    and dataset_review[field]["value"] not in (None, "", [], {})
    and dataset_review[field]["evidence"]
    for field in blocking_fields
)
{
    "project": selected_project,
    "review": dataset_review,
    "blocking_fields": blocking_fields,
    "suitable_for_lab": suitable_for_lab,
    "decision": (
        "continue to adapter design"
        if suitable_for_lab
        else "unsuitable until direct verification"
    ),
}

## Exercise — choose task-shaped metrics

Select one task and add a metric that catches a failure ordinary exact
match would miss. For images, remember that a text-only model cannot see
pixels without a separate OCR or multimodal stage.


In [ ]:
evaluation_plan = {
    "project": selected_project,
    "task_contract": {
        "input": "verified OCR text, not invoice image pixels",
        "output": "strict optional invoice fields with normalized values",
        "authority": "human-reviewed annotations from the verified source",
    },
    "split_risks": [
        "same vendor template crossing splits",
        "duplicate invoice or OCR variants crossing splits",
    ],
    "baselines": [
        "null/empty-field sanity baseline",
        "deterministic pattern-and-normalization extractor",
        "untouched prompted text model",
    ],
    "principal_metrics": [
        "field-level precision/recall/F1",
        "normalized date and currency exactness",
        "hallucinated-field rate",
        "schema validity",
    ],
    "human_review_rule": (
        "route missing, conflicting, or low-confidence critical fields"
    ),
    "modality_boundary": (
        "requires reliable OCR text; a tiny text model cannot read images"
    ),
    "resource_constraints": (
        "verify context-length distribution, latency, and peak memory on "
        "the prepared 24 GB laptop before training"
    ),
}
assert evaluation_plan["project"] == selected_project
evaluation_plan

**Hint:** the output contract determines the evaluator. Classification,
extraction, open-ended transformation, and executable code require
different evidence.


## Final checkpoint

You have completed a full local lifecycle and can now design the next
project without assuming that a public dataset is licensed, suitable,
text-only, balanced, clean, or evaluable in the same way.

Revisit `00_start_here.ipynb` with a new evidence question, version the
source contract, and create a new untouched evaluation boundary.
